# Demand Forecasting — DESD Local Food Marketplace

Predicts weekly product demand for producers using two approaches:

| Model | Description |
|---|---|
| **Baseline** | 4-week simple moving average (SMA) |
| **Improved** | SARIMA(1,1,1)(1,1,1,52) — captures weekly seasonality |

Output: `demand_model.pkl` (fitted SARIMA per product) + `forecast_metrics.json`

**Author: Nazli**

In [1]:
import os, sys, json, warnings, pathlib
warnings.filterwarnings('ignore')

# Django bootstrap — works inside Docker (python manage.py shell) or standalone
sys.path.insert(0, str(pathlib.Path('__file__').resolve().parent.parent.parent))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'backend.settings')
try:
    import django
    django.setup()
    DJANGO_OK = True
except Exception as e:
    print(f'[warn] Django not available: {e}. Using synthetic data only.')
    DJANGO_OK = False

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
import joblib

OUT_DIR = pathlib.Path('.')
MODEL_PATH = OUT_DIR / 'demand_model.pkl'
METRICS_PATH = OUT_DIR / 'forecast_metrics.json'
print('Setup complete.')

[warn] Django not available: No module named 'backend'. Using synthetic data only.


Setup complete.


## 1  Load order history from DB

In [2]:
def load_db_data() -> pd.DataFrame:
    """Pull delivered OrderItems, aggregate to weekly demand per product."""
    from api.models import OrderItem
    rows = (
        OrderItem.objects
        .filter(order__status='Delivered', order__delivery_date__isnull=False)
        .select_related('order', 'product')
        .values('product__id', 'product__name', 'order__delivery_date', 'quantity')
    )
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(list(rows))
    df.columns = ['product_id', 'product_name', 'date', 'quantity']
    df['date'] = pd.to_datetime(df['date'])
    df['week'] = df['date'].dt.to_period('W').apply(lambda r: r.start_time)
    weekly = df.groupby(['product_id', 'product_name', 'week'])['quantity'].sum().reset_index()
    return weekly

if DJANGO_OK:
    db_df = load_db_data()
    print(f'DB rows: {len(db_df)}')
else:
    db_df = pd.DataFrame()
    print('Skipping DB load.')

Skipping DB load.


## 2  Synthetic data (demo mode when DB is sparse)

Generates 52 weeks of realistic weekly demand for each product.  
Signals included: linear trend + annual seasonality + Gaussian noise + occasional harvest spike.

In [3]:
def make_synthetic_data(n_products: int = 8, n_weeks: int = 52, seed: int = 42) -> pd.DataFrame:
    """Generate synthetic weekly demand with trend + seasonality."""
    rng = np.random.default_rng(seed)
    names = [
        'Tomatoes', 'Strawberries', 'Courgettes', 'Apples',
        'Potatoes', 'Carrots', 'Lettuce', 'Raspberries',
    ][:n_products]
    weeks = pd.date_range('2023-10-01', periods=n_weeks, freq='W')
    rows = []
    for pid, name in enumerate(names, 1):
        base   = rng.integers(10, 40)
        trend  = rng.uniform(0.02, 0.15)                   # slight upward trend
        season = rng.uniform(0.3, 0.8)                     # seasonal amplitude
        phase  = rng.uniform(0, 2 * np.pi)                 # random phase offset
        for i, w in enumerate(weeks):
            s   = season * np.sin(2 * np.pi * i / 52 + phase)
            t   = trend * i
            qty = max(0, int(base + t + base * s + rng.normal(0, 2)))
            rows.append({'product_id': pid, 'product_name': name, 'week': w, 'quantity': qty})
    return pd.DataFrame(rows)

MIN_ROWS_FOR_REAL = 20
if len(db_df) >= MIN_ROWS_FOR_REAL:
    df = db_df.copy()
    print(f'Using DB data ({len(df)} rows).')
else:
    df = make_synthetic_data()
    print(f'Using synthetic data ({len(df)} rows). Run with real orders to use DB data.')

products = df[['product_id', 'product_name']].drop_duplicates().sort_values('product_id')
print(products.to_string(index=False))

Using synthetic data (416 rows). Run with real orders to use DB data.
 product_id product_name
          1     Tomatoes
          2 Strawberries
          3   Courgettes
          4       Apples
          5     Potatoes
          6      Carrots
          7      Lettuce
          8  Raspberries


## 3  Train/test split (last 4 weeks = test)

In [4]:
TEST_WEEKS = 4

def split_product(series: pd.Series):
    train = series.iloc[:-TEST_WEEKS]
    test  = series.iloc[-TEST_WEEKS:]
    return train, test

print(f'Test window: last {TEST_WEEKS} weeks.')

Test window: last 4 weeks.


## 4  Baseline — 4-week simple moving average

In [5]:
def moving_average_forecast(train: pd.Series, steps: int, window: int = 4) -> np.ndarray:
    """Predict `steps` future values as the rolling mean of the last `window` observations."""
    last_mean = train.rolling(window, min_periods=1).mean().iloc[-1]
    return np.full(steps, last_mean)

baseline_results = {}
for _, row in products.iterrows():
    pid  = row['product_id']
    name = row['product_name']
    ts   = df[df['product_id'] == pid].set_index('week')['quantity'].sort_index()
    if len(ts) < TEST_WEEKS + 5:
        continue
    train, test = split_product(ts)
    preds = moving_average_forecast(train, TEST_WEEKS)
    mae   = mean_absolute_error(test.values, preds)
    rmse  = mean_squared_error(test.values, preds, squared=False)
    mape  = np.mean(np.abs((test.values - preds) / (test.values + 1e-9))) * 100
    baseline_results[pid] = {'name': name, 'mae': mae, 'rmse': rmse, 'mape': mape}

bdf = pd.DataFrame(baseline_results).T
print('Baseline (4-week SMA) results:')
print(bdf[['name', 'mae', 'rmse', 'mape']].round(2).to_string())

Baseline (4-week SMA) results:
           name   mae       rmse       mape
1      Tomatoes  5.25   5.315073  80.357143
2  Strawberries  4.75   5.267827  19.458898
3    Courgettes   5.0   5.244044  44.185814
4        Apples   2.0   2.291288   4.534611
5      Potatoes   3.0   3.391165  17.118766
6       Carrots   4.5   4.892596  15.266122
7       Lettuce  12.5  13.055171  26.803819
8   Raspberries  5.75   6.600189  37.922209


## 5  Improved model — SARIMA(1,1,1)(1,1,1,52)

Catpures both short-term autocorrelation (p,d,q) and annual weekly seasonality (P,D,Q,52).

In [6]:
def fit_sarima(train: pd.Series, order=(1,1,1), seasonal_order=(1,1,1,52)):
    """Fit SARIMA and return the fitted result."""
    try:
        model = SARIMAX(
            train,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        return model.fit(disp=False)
    except Exception:
        # Fall back to non-seasonal ARIMA if seasonality fitting fails
        model = SARIMAX(train, order=order, enforce_stationarity=False, enforce_invertibility=False)
        return model.fit(disp=False)

sarima_results = {}
fitted_models  = {}   # product_id -> fitted SARIMA for export

for _, row in products.iterrows():
    pid  = row['product_id']
    name = row['product_name']
    ts   = df[df['product_id'] == pid].set_index('week')['quantity'].sort_index()
    if len(ts) < TEST_WEEKS + 5:
        continue
    train, test = split_product(ts)
    # Use simple seasonal order if < 2 full years of data
    seas = (1, 1, 1, 52) if len(train) >= 104 else (0, 0, 0, 0)
    fit  = fit_sarima(train, seasonal_order=seas)
    preds = fit.forecast(steps=TEST_WEEKS)
    preds = np.maximum(preds.values, 0)
    mae   = mean_absolute_error(test.values, preds)
    rmse  = mean_squared_error(test.values, preds, squared=False)
    mape  = np.mean(np.abs((test.values - preds) / (test.values + 1e-9))) * 100
    sarima_results[pid] = {'name': name, 'mae': mae, 'rmse': rmse, 'mape': mape}
    fitted_models[pid]  = {'name': name, 'model': fit, 'last_train_index': train.index[-1]}
    print(f'  {name}: MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.1f}%')

print('\nSARIMA training complete.')

  Tomatoes: MAE=4.36  RMSE=4.45  MAPE=67.1%
  Strawberries: MAE=2.08  RMSE=2.90  MAPE=8.9%
  Courgettes: MAE=5.69  RMSE=5.95  MAPE=50.1%
  Apples: MAE=4.59  RMSE=4.66  MAPE=10.6%
  Potatoes: MAE=3.47  RMSE=4.04  MAPE=20.2%
  Carrots: MAE=4.45  RMSE=4.87  MAPE=15.1%
  Lettuce: MAE=10.12  RMSE=10.78  MAPE=21.6%


  Raspberries: MAE=4.33  RMSE=5.49  MAPE=27.0%



SARIMA training complete.


/usr/local/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency W-SUN will be used.
  

## 6  Comparison

In [7]:
comparison_rows = []
for pid in sorted(set(baseline_results) & set(sarima_results)):
    b = baseline_results[pid]
    s = sarima_results[pid]
    improvement = (b['mae'] - s['mae']) / (b['mae'] + 1e-9) * 100
    comparison_rows.append({
        'product':       b['name'],
        'baseline_mae':  round(b['mae'], 2),
        'sarima_mae':    round(s['mae'], 2),
        'baseline_rmse': round(b['rmse'], 2),
        'sarima_rmse':   round(s['rmse'], 2),
        'improvement_%': round(improvement, 1),
    })

cmp_df = pd.DataFrame(comparison_rows)
print(cmp_df.to_string(index=False))

mean_b_mae = cmp_df['baseline_mae'].mean()
mean_s_mae = cmp_df['sarima_mae'].mean()
print(f'\nMean baseline MAE : {mean_b_mae:.2f}')
print(f'Mean SARIMA MAE   : {mean_s_mae:.2f}')
print(f'Mean improvement  : {(mean_b_mae - mean_s_mae) / mean_b_mae * 100:.1f}%')

     product  baseline_mae  sarima_mae  baseline_rmse  sarima_rmse  improvement_%
    Tomatoes          5.25        4.36           5.32         4.45           16.9
Strawberries          4.75        2.08           5.27         2.90           56.3
  Courgettes          5.00        5.69           5.24         5.95          -13.8
      Apples          2.00        4.59           2.29         4.66         -129.7
    Potatoes          3.00        3.47           3.39         4.04          -15.7
     Carrots          4.50        4.45           4.89         4.87            1.2
     Lettuce         12.50       10.12          13.06        10.78           19.1
 Raspberries          5.75        4.33           6.60         5.49           24.7

Mean baseline MAE : 5.34
Mean SARIMA MAE   : 4.89
Mean improvement  : 8.6%


## 7  Visualise forecasts for top 3 products

In [8]:
top3 = cmp_df.sort_values('sarima_mae').head(3)['product'].tolist()
pid_by_name = {row['product_name']: row['product_id'] for _, row in products.iterrows()}

fig, axes = plt.subplots(1, len(top3), figsize=(5 * len(top3), 4), sharey=False)
if len(top3) == 1:
    axes = [axes]

for ax, name in zip(axes, top3):
    pid = pid_by_name.get(name)
    if pid is None:
        continue
    ts    = df[df['product_id'] == pid].set_index('week')['quantity'].sort_index()
    train, test = split_product(ts)
    # SMA forecast
    sma_pred = moving_average_forecast(train, TEST_WEEKS)
    # SARIMA forecast
    sarima_pred = fitted_models[pid]['model'].forecast(steps=TEST_WEEKS)

    ax.plot(train.index, train.values, color='#1d4ed8', label='Train')
    ax.plot(test.index,  test.values,  color='black',   label='Actual', linewidth=2)
    ax.plot(test.index,  sma_pred,     color='#d97706', linestyle='--', label='SMA baseline')
    ax.plot(test.index,  sarima_pred,  color='#16a34a', linestyle='-',  label='SARIMA')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Week')
    ax.set_ylabel('Units')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Demand Forecast — Baseline vs SARIMA (last 4 weeks)', fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / 'forecast_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved → forecast_comparison.png')

Chart saved → forecast_comparison.png


## 8  Export model + metrics

In [9]:
# Save all fitted SARIMA models keyed by product_id
joblib.dump(fitted_models, MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

# Save metrics JSON
metrics = {
    'generated_at': pd.Timestamp.now().isoformat(),
    'test_weeks': TEST_WEEKS,
    'n_products': len(comparison_rows),
    'mean_baseline_mae': round(mean_b_mae, 4),
    'mean_sarima_mae':   round(mean_s_mae, 4),
    'improvement_pct':   round((mean_b_mae - mean_s_mae) / mean_b_mae * 100, 2),
    'per_product': comparison_rows,
}
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved → {METRICS_PATH}')
print(json.dumps({k: v for k, v in metrics.items() if k != 'per_product'}, indent=2))

Model saved → demand_model.pkl
Metrics saved → forecast_metrics.json
{
  "generated_at": "2026-04-22T19:23:18.107026",
  "test_weeks": 4,
  "n_products": 8,
  "mean_baseline_mae": 5.3438,
  "mean_sarima_mae": 4.8863,
  "improvement_pct": 8.56
}
